In [1]:
# ===============================
# train.py (FULL VERSION + EARLY STOPPING)
# ===============================

import torch
from torch.utils.data import DataLoader
import pickle
import os

from data_preprocessing import load_and_preprocess_data
from tokenizer import BiologicalTokenizer
from model import MiRNADataset, MiRNADiseaseTransformer

# ------------------------------
# 0️⃣ LOG FILE SETUP
# ------------------------------
os.makedirs("logs", exist_ok=True)
log_path = "logs/training_log-100.txt"
log_file = open(log_path, "w")

def log(text):
    print(text)
    log_file.write(text + "\n")

log("==== MiRNA Disease Transformer Training Started ====")

# ------------------------------
# 1️⃣ Collate function
# ------------------------------
def collate_fn(batch):

    max_len = max(len(item['sequence_ids']) for item in batch)

    seqs, masks = [], []
    mirnas, genes = [], []
    disease_labels, context_labels = [], []

    for item in batch:

        seq = item['sequence_ids']
        mask = item['attention_mask']

        pad_len = max_len - len(seq)

        seqs.append(torch.cat([seq, torch.zeros(pad_len, dtype=torch.long)]))
        masks.append(torch.cat([mask, torch.zeros(pad_len, dtype=torch.long)]))

        mirnas.append(item['mirna_ids'])
        genes.append(item['gene_ids'])

        disease_labels.append(item['disease_labels'])
        context_labels.append(item['context_labels'])

    return {
        'sequence_ids': torch.stack(seqs),
        'attention_mask': torch.stack(masks),
        'mirna_ids': torch.stack(mirnas),
        'gene_ids': torch.stack(genes),
        'disease_labels': torch.stack(disease_labels),
        'context_labels': torch.stack(context_labels)
    }


# ------------------------------
# 2️⃣ Load Data
# ------------------------------
train, val, test, dis_enc, ctx_enc = load_and_preprocess_data(
    "data/train_dataset.csv",
    min_samples=2
)

log(f"Train samples: {len(train)}")
log(f"Validation samples: {len(val)}")

# Save encoders
os.makedirs("saved_models", exist_ok=True)
pickle.dump(dis_enc, open("saved_models/disease_encoder.pkl", "wb"))
pickle.dump(ctx_enc, open("saved_models/context_encoder.pkl", "wb"))


# ------------------------------
# 3️⃣ Tokenizer
# ------------------------------
tok = BiologicalTokenizer(train)
tok.save("saved_models/tokenizer.pth")

log("Tokenizer saved.")


# ------------------------------
# 4️⃣ Model
# ------------------------------
model = MiRNADiseaseTransformer(
    vocab_size=len(tok.char_to_idx),
    mirna_vocab=len(tok.mirna_char_to_idx),
    gene_vocab=len(tok.gene_to_idx),
    num_disease=len(dis_enc.classes_),
    num_context=len(ctx_enc.classes_)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

log(f"Using device: {device}")


# ------------------------------
# 5️⃣ Optimizer & Loss
# ------------------------------
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt,
    factor=0.5,
    patience=3
)

loss_fn_disease = torch.nn.CrossEntropyLoss()
loss_fn_context = torch.nn.CrossEntropyLoss()


# ------------------------------
# 6️⃣ DataLoaders
# ------------------------------
train_loader = DataLoader(
    MiRNADataset(train, tok),
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    MiRNADataset(val, tok),
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)


# ------------------------------
# 7️⃣ Early Stopping Setup
# ------------------------------
best_val_loss = float('inf')
patience = 5
counter = 0


# ------------------------------
# 8️⃣ Training Loop
# ------------------------------
num_epochs = 100

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    for b in train_loader:

        seq_ids = b['sequence_ids'].to(device)
        att_mask = b['attention_mask'].to(device)
        mirna_ids = b['mirna_ids'].to(device)
        gene_ids = b['gene_ids'].to(device)

        ctx_labels = b['context_labels'].to(device)
        disease_labels = b['disease_labels'].to(device)

        opt.zero_grad()

        dlog, clog = model(seq_ids, att_mask, mirna_ids, gene_ids, ctx_labels)

        loss_d = loss_fn_disease(dlog, disease_labels)
        loss_c = loss_fn_context(clog, ctx_labels)

        loss = loss_d + loss_c

        loss.backward()
        opt.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    log(f"\nEpoch {epoch+1}/{num_epochs}")
    log(f"Train Loss: {avg_train_loss:.4f}")


    # --------------------------
    # Validation
    # --------------------------
    model.eval()

    val_loss = 0
    correct_d, correct_c, total = 0, 0, 0

    with torch.no_grad():

        for b in val_loader:

            seq_ids = b['sequence_ids'].to(device)
            att_mask = b['attention_mask'].to(device)
            mirna_ids = b['mirna_ids'].to(device)
            gene_ids = b['gene_ids'].to(device)

            ctx_labels = b['context_labels'].to(device)
            disease_labels = b['disease_labels'].to(device)

            dlog, clog = model(seq_ids, att_mask, mirna_ids, gene_ids, ctx_labels)

            loss_d = loss_fn_disease(dlog, disease_labels)
            loss_c = loss_fn_context(clog, ctx_labels)

            loss = loss_d + loss_c
            val_loss += loss.item()

            preds_d = torch.argmax(dlog, dim=1)
            preds_c = torch.argmax(clog, dim=1)

            correct_d += (preds_d == disease_labels).sum().item()
            correct_c += (preds_c == ctx_labels).sum().item()

            total += disease_labels.size(0)

    avg_val_loss = val_loss / len(val_loader)

    disease_acc = correct_d / total
    context_acc = correct_c / total

    log(f"Validation Loss: {avg_val_loss:.4f}")
    log(f"Validation Disease Accuracy: {disease_acc:.4f}")
    log(f"Validation Context Accuracy: {context_acc:.4f}")

    scheduler.step(avg_val_loss)


    # --------------------------
    # Early Stopping Check
    # --------------------------
    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss
        counter = 0

        torch.save(model.state_dict(), "saved_models/best_model.pth")

        log("Best model saved.")

    else:

        counter += 1
        log(f"Early stopping counter: {counter}/{patience}")

        if counter >= patience:
            log("Early stopping triggered.")
            break


# ------------------------------
# 9️⃣ Training Finished
# ------------------------------
log("\nTraining Finished.")

log("Best model saved at saved_models/best_model.pth")

log_file.close()

print(f"\nTraining log saved at: {log_path}")

==== MiRNA Disease Transformer Training Started ====
Train samples: 664
Validation samples: 222
Tokenizer saved.
Using device: cpu


C:\Users\Nabilah\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(



Epoch 1/100
Train Loss: 2.8004
Validation Loss: 2.5095
Validation Disease Accuracy: 0.3829
Validation Context Accuracy: 0.6532
Best model saved.

Epoch 2/100
Train Loss: 2.0078
Validation Loss: 2.1552
Validation Disease Accuracy: 0.5090
Validation Context Accuracy: 0.6712
Best model saved.

Epoch 3/100
Train Loss: 1.6305
Validation Loss: 1.9751
Validation Disease Accuracy: 0.5090
Validation Context Accuracy: 0.6757
Best model saved.

Epoch 4/100
Train Loss: 1.3940
Validation Loss: 1.8360
Validation Disease Accuracy: 0.5856
Validation Context Accuracy: 0.6712
Best model saved.

Epoch 5/100
Train Loss: 1.1938
Validation Loss: 1.7699
Validation Disease Accuracy: 0.5766
Validation Context Accuracy: 0.7027
Best model saved.

Epoch 6/100
Train Loss: 1.0793
Validation Loss: 1.6831
Validation Disease Accuracy: 0.5856
Validation Context Accuracy: 0.6982
Best model saved.

Epoch 7/100
Train Loss: 0.9516
Validation Loss: 1.6341
Validation Disease Accuracy: 0.6036
Validation Context Accuracy: 0.6